# Video Games? No — What Actually Makes Countries Happy 🌍
### An Analytical Study of the World Happiness Report (2015–2023)

**Author:** Kush Prajapati
**Course:** Data Visualization — Final Individual Project, Summer 2026

**Dataset:** World Happiness Report, 2015–2023 (Gallup World Poll / UN Sustainable Development Solutions Network via Kaggle: *World Happiness Report* by `unsdsn`)

This dataset tracks 171 countries across 9 years (2015–2023) and combines:
- **Numerical** — happiness score, GDP per capita, social support, freedom, healthy life expectancy, generosity
- **Categorical** — country, region (11 world regions)
- **Spatial** — country / region
- **Temporal** — year (2015–2023)

The notebook below poses **11 analytical questions** — each one relates two or more variables, compares groups, or tracks change — and answers each with its own explanatory, publication-ready Plotly visualization.

**Design language used throughout:**
- Plotly only, no Matplotlib/Seaborn
- Okabe–Ito colour-blind-safe palette: muted grey for context, one highlight colour for the point being made
- Gridlines and chart-junk removed; direct annotation instead of relying on legends where possible
- Titles state the takeaway, not just the variable names


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Use a renderer that embeds a static-friendly HTML/JS output so charts display correctly
# both inside Jupyter and in PDF/HTML exports of this notebook.
pio.renderers.default = 'notebook_connected'

pd.set_option('display.max_columns', None)

# ---- CVD-safe palette (Okabe-Ito) ----
GREY      = '#BBBBBB'   # context / non-highlighted
BLUE      = '#0072B2'   # primary highlight (positive / focus A)
ORANGE    = '#E69F00'   # secondary highlight (focus B / contrast)
VERMILLION= '#D55E00'   # decline / negative
TEAL      = '#009E73'   # gain / positive
GOLD      = '#F0A202'

TEMPLATE = 'plotly_white'

def style(fig, title, subtitle=None, height=480):
    full_title = f"<b>{title}</b>"
    if subtitle:
        full_title += f"<br><span style='font-size:13px;color:#666'>{subtitle}</span>"
    fig.update_layout(
        template=TEMPLATE,
        title=dict(text=full_title, x=0.02, xanchor='left', font=dict(size=18)),
        font=dict(family='Arial, sans-serif', size=13, color='#222'),
        plot_bgcolor='white', paper_bgcolor='white',
        height=height,
        margin=dict(l=60, r=40, t=90, b=60),
        legend=dict(bgcolor='rgba(0,0,0,0)')
    )
    fig.update_xaxes(showgrid=False, zeroline=False, showline=True, linecolor='#ccc')
    fig.update_yaxes(showgrid=True, gridcolor='#eee', zeroline=False, showline=False)
    return fig


## 0. Data Loading & Cleaning (preliminary — not one of the 10+ analytical questions)

Basic preprocessing: fix a stray region label, check for missing values and duplicate country-year rows, and confirm year coverage.

In [2]:
df = pd.read_csv('World_Happiness_2015_2023.csv')

# --- cleaning ---
# 'Africa' / 'Somaliland region' appears only in 2015-16 for a breakaway territory; fold into Sub-Saharan Africa
df['region'] = df['region'].replace({'Africa': 'Sub-Saharan Africa'})
df['country'] = df['country'].str.strip()

print('Shape:', df.shape)
print('Years covered:', sorted(df.year.unique()))
print('Countries:', df.country.nunique(), '| Regions:', df.region.nunique())
print('\nMissing values:\n', df.isnull().sum())

# one missing healthy_life_expectancy value -> fill with that country's own mean across years
df['healthy_life_expectancy'] = df.groupby('country')['healthy_life_expectancy']\
    .transform(lambda s: s.fillna(s.mean()))

df.head()


Shape: (1367, 9)
Years covered: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Countries: 171 | Regions: 10

Missing values:
 country                         0
region                          0
happiness_score                 0
gdp_per_capita                  0
generosity                      0
social_support                  0
freedom_to_make_life_choices    0
healthy_life_expectancy         1
year                            0
dtype: int64


,country,region,happiness_score,gdp_per_capita,generosity,social_support,freedom_to_make_life_choices,healthy_life_expectancy,year
0,Switzerland,Western Europe,7.587,1.39651,0.29678,1.34951,0.66557,0.94143,2015
1,Iceland,Western Europe,7.561,1.30232,0.43630,1.40223,0.62877,0.94784,2015
2,Denmark,Western Europe,7.527,1.32548,0.34139,1.36058,0.64938,0.87464,2015
3,Norway,Western Europe,7.522,1.45900,0.34699,1.33095,0.66973,0.88521,2015
4,Canada,North America and ANZ,7.427,1.32629,0.45811,1.32261,0.63297,0.90563,2015


**Preliminary EDA note:** the checks above (shape, missing values, year coverage) are just data-quality housekeeping — they are *not* among the 10+ analytical questions below, which each combine multiple variables to produce a genuine insight.

---
## Q1. How has the happiness gap between the best- and worst-performing regions changed between 2015 and 2023?

*(Trend across time, compared across groups)*

In [3]:
reg_year = df.groupby(['year','region'], as_index=False)['happiness_score'].mean()

order_2023 = reg_year[reg_year.year==2023].sort_values('happiness_score')['region']
best_region  = order_2023.iloc[-1]
worst_region = order_2023.iloc[0]

gap_2015 = reg_year[(reg_year.year==2015)]['happiness_score'].max() - reg_year[(reg_year.year==2015)]['happiness_score'].min()
gap_2023 = reg_year[(reg_year.year==2023)]['happiness_score'].max() - reg_year[(reg_year.year==2023)]['happiness_score'].min()

fig = go.Figure()
for region in reg_year.region.unique():
    sub = reg_year[reg_year.region == region]
    if region == best_region:
        fig.add_trace(go.Scatter(x=sub.year, y=sub.happiness_score, mode='lines',
                                  line=dict(color=TEAL, width=3), name=region))
    elif region == worst_region:
        fig.add_trace(go.Scatter(x=sub.year, y=sub.happiness_score, mode='lines',
                                  line=dict(color=VERMILLION, width=3), name=region))
    else:
        fig.add_trace(go.Scatter(x=sub.year, y=sub.happiness_score, mode='lines',
                                  line=dict(color=GREY, width=1.4), name=region, opacity=0.6,
                                  showlegend=False))

fig.add_annotation(x=2023, y=reg_year[(reg_year.year==2023)&(reg_year.region==best_region)]['happiness_score'].values[0],
                    text=f'{best_region}<br>(happiest, 2023)', showarrow=True, arrowhead=2, ax=40, ay=-30, font=dict(color=TEAL, size=11))
fig.add_annotation(x=2023, y=reg_year[(reg_year.year==2023)&(reg_year.region==worst_region)]['happiness_score'].values[0],
                    text=f'{worst_region}<br>(least happy, 2023)', showarrow=True, arrowhead=2, ax=40, ay=30, font=dict(color=VERMILLION, size=11))

fig = style(fig, f"The happiness gap between regions narrowed slightly: {gap_2015:.2f} pts (2015) → {gap_2023:.2f} pts (2023)",
            "Average happiness score by region, 2015–2023 — top and bottom region highlighted")
fig.show()

print(f'Gap 2015: {gap_2015:.2f} | Gap 2023: {gap_2023:.2f}')


Gap 2015: 3.08 | Gap 2023: 2.93


---
## Q2. Does GDP per capita drive happiness equally in every region — or does money matter more in some places than others?

*(Relationship between two variables, conditioned on a third — region)*

In [4]:
corr_by_region = df.groupby('region').apply(lambda g: g['gdp_per_capita'].corr(g['happiness_score']), include_groups=False)\
    .sort_values().reset_index()
corr_by_region.columns = ['region', 'corr']

colors = [VERMILLION if v < 0 else (BLUE if v == corr_by_region['corr'].max() else GREY) for v in corr_by_region['corr']]

fig = go.Figure(go.Bar(
    x=corr_by_region['corr'], y=corr_by_region['region'], orientation='h',
    marker_color=colors,
    text=[f'{v:.2f}' for v in corr_by_region['corr']], textposition='outside'
))
fig.add_vline(x=0, line_color='#999', line_width=1)

weakest = corr_by_region.iloc[0]
strongest = corr_by_region.iloc[-1]
fig.add_annotation(x=weakest['corr'], y=weakest['region'],
                    text='Only region where higher GDP does NOT track with higher happiness',
                    showarrow=True, arrowhead=2, ax=60, ay=-25, font=dict(color=VERMILLION, size=11))

fig = style(fig, f"Wealth buys happiness almost everywhere — except {weakest['region']}",
            "Correlation between GDP per capita and happiness score, by region (2015–2023 pooled)", height=520)
fig.update_xaxes(title='Correlation coefficient (GDP per capita vs. happiness score)')
fig.update_yaxes(title=None)
fig.show()


---
## Q3. Which regions gained the most — and lost the most — average happiness between 2015 and 2023?

*(Change over time, compared across groups)*

In [5]:
piv = reg_year.pivot(index='region', columns='year', values='happiness_score')
change = (piv[2023] - piv[2015]).sort_values()

colors = [VERMILLION if v < 0 else TEAL for v in change]

fig = go.Figure(go.Bar(
    x=change.values, y=change.index, orientation='h', marker_color=colors,
    text=[f'{v:+.2f}' for v in change.values], textposition='outside'
))
fig.add_vline(x=0, line_color='#999', line_width=1)
fig = style(fig, "Central & Eastern Europe climbed the most; the Middle East & North Africa slid the most",
            "Change in average regional happiness score, 2015 → 2023", height=520)
fig.update_xaxes(title='Change in happiness score (2023 − 2015)')
fig.update_yaxes(title=None)
fig.show()


---
## Q4. In 2023, how do the five underlying happiness factors differ between the world's happiest and least-happy regions?

*(Multi-variable comparison across groups)*

In [6]:
factors = ['gdp_per_capita','social_support','freedom_to_make_life_choices','healthy_life_expectancy','generosity']
labels  = ['GDP per capita','Social support','Freedom','Healthy life exp.','Generosity']

d2023 = df[df.year==2023]
top3_regions = d2023.groupby('region')['happiness_score'].mean().nlargest(3).index
bot3_regions = d2023.groupby('region')['happiness_score'].mean().nsmallest(3).index

top_vals = d2023[d2023.region.isin(top3_regions)][factors].mean()
bot_vals = d2023[d2023.region.isin(bot3_regions)][factors].mean()

# normalise each factor to 0-1 across the whole 2023 sample so the radar is comparable
norm = d2023[factors].copy()
norm = (norm - norm.min()) / (norm.max() - norm.min())
top_n = norm.loc[d2023.region.isin(top3_regions)].mean()
bot_n = norm.loc[d2023.region.isin(bot3_regions)].mean()

fig = go.Figure()
fig.add_trace(go.Scatterpolar(r=list(top_n)+[top_n.iloc[0]], theta=labels+[labels[0]],
                               fill='toself', name='Top-3 happiest regions', line=dict(color=BLUE), fillcolor='rgba(0,114,178,0.25)'))
fig.add_trace(go.Scatterpolar(r=list(bot_n)+[bot_n.iloc[0]], theta=labels+[labels[0]],
                               fill='toself', name='Bottom-3 happiest regions', line=dict(color=VERMILLION), fillcolor='rgba(213,94,0,0.20)'))

fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0,1], showticklabels=False, gridcolor='#eee')))
fig = style(fig, "The happiest regions don't just out-earn the least happy ones — they out-support and out-trust them",
            "Normalised 2023 factor averages: top-3 vs bottom-3 happiest regions", height=520)
fig.show()


---
## Q5. Is generosity's link to happiness getting stronger or weaker over time?

*(Relationship tracked over time)*

In [7]:
gen_corr = df.groupby('year').apply(lambda g: g['generosity'].corr(g['happiness_score']), include_groups=False)

colors = [VERMILLION if v < 0 else BLUE for v in gen_corr]
fig = go.Figure(go.Bar(x=gen_corr.index, y=gen_corr.values, marker_color=colors))
fig.add_hline(y=0, line_color='#999')
fig.add_annotation(x=2021, y=gen_corr.loc[2021], text='briefly turned negative<br>during 2021',
                    showarrow=True, arrowhead=2, ay=-40, font=dict(color=VERMILLION, size=11))
fig = style(fig, "Generosity's link to happiness has weakened steadily since 2015",
            "Correlation between generosity and happiness score, by year")
fig.update_xaxes(dtick=1, title='Year')
fig.update_yaxes(title='Correlation coefficient')
fig.show()


---
## Q6. Which individual countries saw the biggest happiness gains and losses from 2015 to 2023?

*(Change over time, at country level — grounding the regional story in real places)*

In [8]:
c15 = df[df.year==2015].set_index('country')['happiness_score']
c23 = df[df.year==2023].set_index('country')['happiness_score']
common = c15.index.intersection(c23.index)
change_c = (c23[common] - c15[common]).sort_values()

top_gain = change_c.tail(5)
top_loss = change_c.head(5)
combo = pd.concat([top_loss, top_gain])
colors = [VERMILLION]*5 + [TEAL]*5

fig = go.Figure(go.Bar(x=combo.values, y=combo.index, orientation='h', marker_color=colors,
                        text=[f'{v:+.2f}' for v in combo.values], textposition='outside'))
fig.add_vline(x=0, line_color='#999')
fig.add_annotation(x=top_loss.iloc[0], y=top_loss.index[0], text='Economic & political crisis years',
                    showarrow=True, arrowhead=2, ax=60, ay=0, font=dict(color=VERMILLION, size=11))
fig = style(fig, f"{top_loss.index[0]} fell the furthest; {top_gain.index[-1]} climbed the most (2015→2023)",
            "5 biggest happiness-score declines vs. 5 biggest gains, by country", height=520)
fig.update_xaxes(title='Change in happiness score (2023 − 2015)')
fig.update_yaxes(title=None)
fig.show()


---
## Q7. Has the strength of the link between healthy life expectancy and happiness increased or decreased since 2015?

*(Relationship tracked over time)*

In [9]:
health_corr = df.groupby('year').apply(lambda g: g['healthy_life_expectancy'].corr(g['happiness_score']), include_groups=False)

fig = go.Figure()
fig.add_trace(go.Scatter(x=health_corr.index, y=health_corr.values, mode='lines+markers',
                          line=dict(color=BLUE, width=3), marker=dict(size=8)))
fig.add_annotation(x=2017, y=health_corr.loc[2017], text='peak: 0.78',
                    showarrow=True, arrowhead=2, ay=-30, font=dict(size=11, color=BLUE))
fig = style(fig, "Health has stayed one of the strongest, steadiest predictors of happiness (r ≈ 0.72–0.78 every year)",
            "Correlation between healthy life expectancy and happiness score, by year")
fig.update_yaxes(range=[0,1], title='Correlation coefficient')
fig.update_xaxes(dtick=1, title='Year')
fig.show()


---
## Q8. Do countries with high freedom to make life choices — but only modest GDP — still manage to be happy?

*(Interaction between two variables, with a third variable — GDP — shown as bubble size)*

In [10]:
d2023 = df[df.year==2023].copy()
median_gdp = d2023['gdp_per_capita'].median()

# highlight countries: high freedom (top tercile) + below-median GDP
freedom_hi = d2023['freedom_to_make_life_choices'].quantile(0.66)
mask = (d2023['freedom_to_make_life_choices'] >= freedom_hi) & (d2023['gdp_per_capita'] < median_gdp)

fig = go.Figure()
fig.add_trace(go.Scatter(x=d2023.loc[~mask,'freedom_to_make_life_choices'], y=d2023.loc[~mask,'happiness_score'],
                          mode='markers', marker=dict(size=d2023.loc[~mask,'gdp_per_capita']*14+4, color=GREY, opacity=0.5),
                          name='Other countries', showlegend=False))
fig.add_trace(go.Scatter(x=d2023.loc[mask,'freedom_to_make_life_choices'], y=d2023.loc[mask,'happiness_score'],
                          mode='markers+text', text=d2023.loc[mask,'country'], textposition='top center',
                          marker=dict(size=d2023.loc[mask,'gdp_per_capita']*14+4, color=TEAL, line=dict(color='white',width=1)),
                          name='High freedom, below-median GDP', textfont=dict(size=9)))

fig = style(fig, "High freedom can partly substitute for GDP — several below-median-income countries still rank happy",
            "Freedom to make life choices vs. happiness score, 2023 (bubble size = GDP per capita)", height=520)
fig.update_xaxes(title='Freedom to make life choices')
fig.update_yaxes(title='Happiness score')
fig.show()

print(d2023.loc[mask, ['country','freedom_to_make_life_choices','gdp_per_capita','happiness_score']]
      .sort_values('happiness_score', ascending=False).head(8))


          country  freedom_to_make_life_choices  gdp_per_capita  \
1263       Kosovo                         0.639           1.374   
1269    Nicaragua                         0.660           1.109   
1272    Guatemala                         0.631           1.287   
1279  El Salvador                         0.713           1.278   
1283   Uzbekistan                         0.740           1.227   
1291   Kyrgyzstan                         0.735           1.061   
1294      Vietnam                         0.741           1.349   
1295     Paraguay                         0.678           1.428   

      happiness_score  
1263            6.368  
1269            6.259  
1272            6.150  
1279            6.122  
1283            6.014  
1291            5.825  
1294            5.763  
1295            5.738  


---
## Q9. Do wealthier regions also have stronger social bonds — or are wealth and social support independent?

*(Relationship between two variables at the regional level)*

In [11]:
reg_2023 = df[df.year==2023].groupby('region', as_index=False).agg(
    gdp=('gdp_per_capita','mean'), social=('social_support','mean'), happy=('happiness_score','mean'))

corr_val = reg_2023['gdp'].corr(reg_2023['social'])

fig = go.Figure(go.Scatter(
    x=reg_2023['gdp'], y=reg_2023['social'], mode='markers+text',
    text=reg_2023['region'], textposition='top center', textfont=dict(size=10),
    marker=dict(size=reg_2023['happy']*8, color=BLUE, opacity=0.75, line=dict(color='white', width=1))
))
fig = style(fig, f"Richer regions do report stronger social support too (r = {corr_val:.2f}) — wealth and social bonds move together",
            "Regional average GDP per capita vs. social support, 2023 (bubble size = happiness score)", height=520)
fig.update_xaxes(title='GDP per capita (regional avg.)')
fig.update_yaxes(title='Social support (regional avg.)')
fig.show()


---
## Q10. Which of the five happiness factors is most consistently linked to happiness across all nine years?

*(Multi-variable comparison, tracked over time — heatmap)*

In [12]:
factors = ['gdp_per_capita','social_support','freedom_to_make_life_choices','healthy_life_expectancy','generosity']
factor_labels = ['GDP per capita','Social support','Freedom','Healthy life exp.','Generosity']

corr_table = df.groupby('year').apply(
    lambda g: pd.Series({f: g[f].corr(g['happiness_score']) for f in factors}), include_groups=False
)
corr_table.columns = factor_labels

fig = go.Figure(go.Heatmap(
    z=corr_table.values.T, x=corr_table.index, y=factor_labels,
    colorscale='Blues', zmin=-0.2, zmax=1,
    text=np.round(corr_table.values.T,2), texttemplate='%{text}', textfont=dict(size=11),
    colorbar=dict(title='corr.')
))
fig = style(fig, "GDP and social support are consistently the strongest drivers of happiness — generosity barely registers",
            "Correlation of each factor with happiness score, by year", height=460)
fig.update_xaxes(dtick=1, title='Year')
fig.update_yaxes(title=None)
fig.show()


---
## Q11. Which regions are the most volatile in happiness year-to-year, and which are the most stable?

*(Variability over time, compared across groups)*

In [13]:
vol = df.groupby('region')['happiness_score'].std().sort_values()

colors = [BLUE if r==vol.index[0] else (VERMILLION if r==vol.index[-1] else GREY) for r in vol.index]

fig = go.Figure(go.Bar(x=vol.values, y=vol.index, orientation='h', marker_color=colors,
                        text=[f'{v:.2f}' for v in vol.values], textposition='outside'))
fig.add_annotation(x=vol.iloc[-1], y=vol.index[-1], text='Most volatile — swings with regional conflict & crises',
                    showarrow=True, arrowhead=2, ax=70, ay=0, font=dict(color=VERMILLION, size=11))
fig.add_annotation(x=vol.iloc[0], y=vol.index[0], text='Most stable region',
                    showarrow=True, arrowhead=2, ax=60, ay=0, font=dict(color=BLUE, size=11))
fig = style(fig, "North America & ANZ is remarkably stable; the Middle East & North Africa swings the most",
            "Standard deviation of happiness score within each region, 2015–2023", height=520)
fig.update_xaxes(title='Std. deviation of happiness score')
fig.update_yaxes(title=None)
fig.show()


---
## Key Insights & Conclusions

1. **The global happiness gap is narrowing slightly** — the best- and worst-performing regions are closer together in 2023 than in 2015, though the gap remains substantial.
2. **Money helps almost everywhere except in already-wealthy, stable regions.** GDP per capita correlates strongly and positively with happiness in most regions, but the relationship is weakest (even slightly negative) in North America & ANZ, where income has plateaued while happiness varies for other reasons.
3. **Regional trajectories diverge sharply.** Central & Eastern Europe posted the strongest average gains 2015→2023, while the Middle East & North Africa saw the steepest average decline — a pattern echoed at the country level, where Lebanon and Afghanistan fell the furthest and Romania climbed the most.
4. **It's not just GDP that separates happy regions from unhappy ones** — the happiest regions also report meaningfully higher social support, freedom, and healthy life expectancy, not just higher income.
5. **Generosity's link to happiness has quietly weakened** across almost every year of the dataset, briefly turning negative in 2021, while healthy life expectancy has remained one of the steadiest, strongest predictors throughout.
6. **Freedom can partly compensate for lower income** — several countries with below-median GDP but high freedom to make life choices still report high happiness.
7. **Wealth and social support move together at the regional level**, suggesting they reinforce rather than substitute for each other.
8. **GDP and social support are the most consistently powerful drivers of happiness year after year**, while generosity is the weakest and least reliable factor.
9. **Some regions are simply more volatile than others** — the Middle East & North Africa swings the most year-to-year, while North America & ANZ is the most stable, reflecting differences in political and economic shocks.

**Dataset:** `World_Happiness_2015_2023.csv` — combined World Happiness Report data (2015–2023), sourced from Kaggle (originally Gallup World Poll / UN Sustainable Development Solutions Network).
